# Sortformer diarization experiment (Google Colab)

Diarizes one interview with NVIDIA Sortformer (NeMo) and writes the same JSON
shape as `src/diarization/pyannote_runner.py`, so it can be scored locally:

```powershell
.\.venv\Scripts\python.exe -X utf8 -m scripts.compare_diarization --checkpoints <sample>/diarization_checkpoints.json <sample>/interview_16k_mono_diarization.json <sample>/interview_16k_mono_diarization_sortformer.json
```

**Input (upload to `/content`):** `interview_16k_mono.wav` (16 kHz mono).

**Output (downloaded):** `interview_16k_mono_diarization_sortformer.json`.
Put it next to the existing diarization file; do not overwrite it.

**Privacy:** clear all outputs before saving this notebook into the repository.
Check the model card licence before any use beyond this experiment.


In [ ]:
!pip install -q "nemo_toolkit[asr]"

In [ ]:
from pathlib import Path

AUDIO_PATH = Path("/content/interview_16k_mono.wav")
OUTPUT_PATH = Path("/content/interview_16k_mono_diarization_sortformer.json")

# Response ID of the sample, used as audio_id in the output.
AUDIO_ID = "SET_RESPONSE_ID"

# Streaming Sortformer handles long recordings; v1 is the offline model.
MODEL_NAME = "nvidia/diar_streaming_sortformer_4spk-v2"

In [ ]:
import soundfile as sf
from google.colab import files

if not AUDIO_PATH.exists():
    print(f"Upload {AUDIO_PATH.name}")
    files.upload()

if not AUDIO_PATH.exists():
    raise FileNotFoundError(f"{AUDIO_PATH.name} was not uploaded.")

if AUDIO_ID == "SET_RESPONSE_ID":
    raise ValueError("Set AUDIO_ID to the sample's response ID.")

info = sf.info(str(AUDIO_PATH))
print(f"Duration: {info.duration:.1f}s  Sample rate: {info.samplerate}  Channels: {info.channels}")

if info.samplerate != 16000 or info.channels != 1:
    raise ValueError("Expected 16 kHz mono WAV (interview_16k_mono.wav).")

In [ ]:
import torch
import nemo
from nemo.collections.asr.models import SortformerEncLabelModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if DEVICE == "cpu":
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU.")

diar_model = SortformerEncLabelModel.from_pretrained(MODEL_NAME)
diar_model = diar_model.to(DEVICE)
diar_model.eval()

# Streaming settings from the model card (high-latency, most accurate).
modules = getattr(diar_model, "sortformer_modules", None)
streaming = {
    "chunk_len": 340,
    "chunk_right_context": 40,
    "fifo_len": 40,
    "spkcache_update_period": 300,
}
if modules is not None:
    for name, value in streaming.items():
        if hasattr(modules, name):
            setattr(modules, name, value)

print(f"Model: {MODEL_NAME}  NeMo: {nemo.__version__}  Device: {DEVICE}")

In [ ]:
import datetime
import json


def parse_segment(item):
    # NeMo returns "start end speaker_N" strings; accept tuples too.
    if isinstance(item, str):
        start, end, speaker = item.split()[:3]
    else:
        start, end, speaker = list(item)[:3]
    return float(start), float(end), str(speaker)


outputs = diar_model.diarize(audio=[str(AUDIO_PATH)], batch_size=1)

# One list of segments per input file; some versions return a tuple.
if isinstance(outputs, tuple):
    outputs = outputs[0]
raw_segments = outputs[0]

segments = []
for item in raw_segments:
    start, end, speaker = parse_segment(item)
    if end > start:
        segments.append({
            "speaker_id": speaker,
            "start_sec": round(start, 3),
            "end_sec": round(end, 3),
        })

segments.sort(key=lambda s: (s["start_sec"], s["end_sec"]))

result = {
    "audio_id": AUDIO_ID,
    "model_name": MODEL_NAME,
    "created_at_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "nemo_version": nemo.__version__,
    "segments": segments,
}

OUTPUT_PATH.write_text(json.dumps(result, indent=2), encoding="utf-8")

totals = {}
for s in segments:
    totals[s["speaker_id"]] = totals.get(s["speaker_id"], 0) + s["end_sec"] - s["start_sec"]

print(f"Segments: {len(segments)}")
print("Speaker totals (s):", {k: round(v, 1) for k, v in sorted(totals.items())})

files.download(str(OUTPUT_PATH))